# Étape 3 — RQ1 : premiers résultats et pilote LLM

Ce notebook fait trois choses :

1. **Construit le « corrigé »** : les 4 865 paires de MOOCCubeX annotées par des humains (`ground_truth` = 1 ou 0).
2. **Calcule votre premier tableau de résultats** : la qualité des deux modèles classiques fournis avec MOOCCubeX (`text_predict`, `graph_predict`).
3. **Vérifie Junyi 2015** : nombre de prérequis d'experts et de réponses d'élèves sur ces exercices.
4. **Lance un pilote LLM** (Qwen, open-source, sur le GPU de Colab) sur 300 paires, avec 3 formulations de question.

**Avant de commencer :** menu *Exécution → Modifier le type d'exécution → GPU T4*. Le GPU n'est utile que pour les cellules 5 et 6.

In [ ]:
# Cellule 1 — Drive, chemins et outils communs
from google.colab import drive
drive.mount('/content/drive')

import os, json, random
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, roc_auc_score

DOSSIER   = '/content/drive/MyDrive/projet_kt'
PREREQ    = os.path.join(DOSSIER, 'prerequisites')
JUNYI     = os.path.join(DOSSIER, 'junyi2015')
RESULTATS = os.path.join(DOSSIER, 'resultats')
os.makedirs(RESULTATS, exist_ok=True)
DOMAINES  = {'cs': 'cs.json', 'math': 'math.json', 'psy': 'psy.json'}
GRAINE = 42

def metriques(y_vrai, score, seuil=0.5):
    # Précision, rappel, F1, exactitude au seuil, et AUC (indépendante du seuil)
    y_vrai = np.asarray(y_vrai); score = np.asarray(score, dtype=float)
    y_pred = (score >= seuil).astype(int)
    return {
        'n': len(y_vrai),
        'precision': round(precision_score(y_vrai, y_pred, zero_division=0), 3),
        'rappel':    round(recall_score(y_vrai, y_pred, zero_division=0), 3),
        'F1':        round(f1_score(y_vrai, y_pred, zero_division=0), 3),
        'exactitude': round(accuracy_score(y_vrai, y_pred), 3),
        'AUC': round(roc_auc_score(y_vrai, score), 3) if len(set(y_vrai)) == 2 else None,
    }
print('OK')

In [ ]:
# Cellule 2 — Le « corrigé » : paires de MOOCCubeX annotées par des humains
lignes = []
for dom, nom in DOMAINES.items():
    with open(os.path.join(PREREQ, nom), encoding='utf-8') as f:
        for l in f:
            if not l.strip():
                continue
            r = json.loads(l)
            if r.get('ground_truth') in (0, 1):
                tp, gp = r.get('text_predict'), r.get('graph_predict')
                lignes.append({
                    'domaine': dom, 'A': r['c1'], 'B': r['c2'], 'label': r['ground_truth'],
                    # 2e valeur = probabilité « A est prérequis de B »
                    'score_texte': tp[1] if isinstance(tp, list) and len(tp) == 2 else np.nan,
                    'score_graphe': gp[1] if isinstance(gp, list) and len(gp) == 2 else np.nan,
                })
bench = pd.DataFrame(lignes).drop_duplicates(subset=['domaine', 'A', 'B'])
bench.to_csv(os.path.join(RESULTATS, 'benchmark_mooccubex.csv'), index=False)
print(bench.groupby(['domaine', 'label']).size().unstack())
print('Total :', len(bench))
bench.head()

In [ ]:
# Cellule 3 — Premier tableau de résultats : les deux modèles classiques de MOOCCubeX
res = []
for modele, col in [('Classique texte', 'score_texte'), ('Classique graphe', 'score_graphe')]:
    for dom in ['TOUT'] + list(DOMAINES):
        sous = bench if dom == 'TOUT' else bench[bench.domaine == dom]
        sous = sous.dropna(subset=[col])
        res.append({'modele': modele, 'domaine': dom, **metriques(sous.label, sous[col])})
tab_classiques = pd.DataFrame(res)
tab_classiques.to_csv(os.path.join(RESULTATS, 'rq1_modeles_classiques.csv'), index=False)
tab_classiques

In [ ]:
# Cellule 4 — Junyi 2015 : prérequis d'experts et réponses d'élèves
exo = pd.read_csv(os.path.join(JUNYI, 'junyi_Exercise_table.csv'))
aretes = []
for _, r in exo.iterrows():
    if isinstance(r['prerequisites'], str) and r['prerequisites'].strip():
        for p in r['prerequisites'].split(','):
            if p.strip():
                aretes.append((p.strip(), r['name']))
aretes = pd.DataFrame(aretes, columns=['A', 'B']).drop_duplicates()
noms_exo = set(exo['name'])
print('Exercices :', len(exo), '| avec au moins un prérequis :', aretes.B.nunique())
print('Arêtes de prérequis (expert) :', len(aretes),
      '| dont les deux exercices existent dans la table :', int((aretes.A.isin(noms_exo) & aretes.B.isin(noms_exo)).sum()))
aretes.to_csv(os.path.join(RESULTATS, 'junyi_aretes_expert.csv'), index=False)

# Réponses : lecture par morceaux (2,5 Go), en ne gardant que 3 colonnes
chemin_log = os.path.join(JUNYI, 'junyi_ProblemLog_original.csv')
entete = pd.read_csv(chemin_log, nrows=0).columns.tolist()
print('Colonnes du journal :', entete)
col_u = 'user_id' if 'user_id' in entete else entete[0]
col_e = 'exercise' if 'exercise' in entete else next(c for c in entete if 'exercise' in c)
col_c = 'correct' if 'correct' in entete else next(c for c in entete if 'correct' in c)

exos_aretes = set(aretes.A) | set(aretes.B)
n_total = n_sur_aretes = n_corrects = 0
eleves, rep_par_exo = set(), Counter()
for morceau in pd.read_csv(chemin_log, usecols=[col_u, col_e, col_c], chunksize=2_000_000):
    n_total += len(morceau)
    m = morceau[morceau[col_e].isin(exos_aretes)]
    n_sur_aretes += len(m)
    n_corrects += m[col_c].astype(str).str.lower().isin(['true', '1']).sum()
    eleves.update(m[col_u].unique())
    rep_par_exo.update(m[col_e].value_counts().to_dict())

stats_junyi = {
    'reponses_total': n_total,
    'reponses_sur_exercices_avec_aretes': n_sur_aretes,
    'eleves_concernes': len(eleves),
    'taux_reussite': round(n_corrects / max(1, n_sur_aretes), 3),
    'aretes_dont_les_2_exercices_ont_des_reponses': int(sum(1 for a, b in aretes.itertuples(index=False)
                                                           if rep_par_exo[a] > 0 and rep_par_exo[b] > 0)),
}
for k, v in stats_junyi.items():
    print(f'{k:48s} {v:,}' if isinstance(v, int) else f'{k:48s} {v}')

## Pilote LLM

**Méthode :** pour chaque paire (A, B), on pose une question fermée au LLM et on lit directement la probabilité qu'il réponde « oui ».
On n'a donc pas besoin de faire écrire de texte au modèle : c'est rapide, et on obtient un score entre 0 et 1, comparable aux modèles classiques.

On teste **3 formulations** de la question, et on pose aussi la question **dans l'autre sens** (B → A) pour mesurer les **inversions** (une erreur importante de votre taxonomie).

Modèle : **Qwen2.5-7B-Instruct** compressé en 4 bits pour tenir sur le GPU T4 gratuit. Si la mémoire manque, mettez `MODELE = 'Qwen/Qwen2.5-3B-Instruct'`.

In [ ]:
# Cellule 5 — Chargement du LLM (5 à 10 min la première fois)
!pip install -q -U transformers accelerate bitsandbytes
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

assert torch.cuda.is_available(), "Pas de GPU : Exécution → Modifier le type d'exécution → GPU T4"
MODELE = 'Qwen/Qwen2.5-7B-Instruct'
tok = AutoTokenizer.from_pretrained(MODELE)
tok.padding_side = 'left'
llm = AutoModelForCausalLM.from_pretrained(
    MODELE, device_map='auto',
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16),
)
llm.eval()

NOMS_DOMAINE = {'cs': '计算机科学', 'math': '数学', 'psy': '心理学'}
PROMPTS = {
    # P1 : question directe, en chinois
    'P1_direct_zh': ('你是{dom}领域的教育专家。', '概念A：{A}\n概念B：{B}\n概念A是概念B的先修知识吗？只回答“是”或“否”。', ('是', '否')),
    # P2 : formulation pédagogique (faut-il maîtriser A avant d'apprendre B ?)
    'P2_pedago_zh': ('你是{dom}课程设计专家。', '学生在学习“{B}”之前，是否必须先掌握“{A}”？只回答“是”或“否”。', ('是', '否')),
    # P3 : même question en anglais
    'P3_direct_en': ('You are an expert teacher in {dom}.', 'Concept A: {A}\nConcept B: {B}\nIs concept A a prerequisite for learning concept B? Answer only Yes or No.', ('Yes', 'No')),
}

def ids_reponse(oui, non):
    return tok.encode(oui, add_special_tokens=False)[0], tok.encode(non, add_special_tokens=False)[0]

@torch.no_grad()
def scores_oui(paires, cle_prompt, taille_lot=16):
    systeme, gabarit, (oui, non) = PROMPTS[cle_prompt]
    i_oui, i_non = ids_reponse(oui, non)
    textes = [tok.apply_chat_template(
                  [{'role': 'system', 'content': systeme.format(dom=NOMS_DOMAINE[d])},
                   {'role': 'user', 'content': gabarit.format(A=a, B=b)}],
                  tokenize=False, add_generation_prompt=True)
              for d, a, b in paires]
    sortie = []
    for i in range(0, len(textes), taille_lot):
        lot = tok(textes[i:i + taille_lot], return_tensors='pt', padding=True).to(llm.device)
        logits = llm(**lot).logits[:, -1, :].float()
        deux = torch.stack([logits[:, i_oui], logits[:, i_non]], dim=1)
        sortie.extend(torch.softmax(deux, dim=1)[:, 0].cpu().tolist())
    return sortie

print('Modèle chargé :', MODELE)
print('Test :', scores_oui([('math', '函数的定义域', '欧拉函数')], 'P1_direct_zh'))

In [ ]:
# Cellule 6 — Pilote : 300 paires (50 « oui » + 50 « non » par domaine), 3 prompts, 2 sens
import time
pilote = pd.concat([g.sample(min(50, len(g)), random_state=GRAINE)
                    for _, g in bench.groupby(['domaine', 'label'])]).reset_index(drop=True)
print('Paires du pilote :', len(pilote))

t0 = time.time()
for cle in PROMPTS:
    pilote[f'{cle}_AB'] = scores_oui(list(zip(pilote.domaine, pilote.A, pilote.B)), cle)
    pilote[f'{cle}_BA'] = scores_oui(list(zip(pilote.domaine, pilote.B, pilote.A)), cle)
    print(f'{cle} terminé ({(time.time() - t0) / 60:.1f} min)')
pilote.to_csv(os.path.join(RESULTATS, 'rq1_pilote_scores.csv'), index=False)

res = []
for cle in PROMPTS:
    for dom in ['TOUT'] + list(DOMAINES):
        sous = pilote if dom == 'TOUT' else pilote[pilote.domaine == dom]
        positifs = sous[sous.label == 1]
        res.append({
            'modele': f'{MODELE.split("/")[-1]} {cle}', 'domaine': dom,
            **metriques(sous.label, sous[f'{cle}_AB']),
            # Inversion : sur les vrais prérequis A→B, le LLM dit aussi « oui » pour B→A
            'taux_inversion': round(float((positifs[f'{cle}_BA'] >= 0.5).mean()), 3) if len(positifs) else None,
        })
# Modèles classiques sur les MÊMES 300 paires, pour une comparaison équitable
for modele, col in [('Classique texte', 'score_texte'), ('Classique graphe', 'score_graphe')]:
    for dom in ['TOUT'] + list(DOMAINES):
        sous = (pilote if dom == 'TOUT' else pilote[pilote.domaine == dom]).dropna(subset=[col])
        res.append({'modele': modele, 'domaine': dom, **metriques(sous.label, sous[col]), 'taux_inversion': None})
tab_pilote = pd.DataFrame(res)
tab_pilote.to_csv(os.path.join(RESULTATS, 'rq1_pilote_resultats.csv'), index=False)
tab_pilote[tab_pilote.domaine == 'TOUT']

In [ ]:
# Cellule 7 — Résumé à copier dans la conversation
print('=== Modèles classiques (toutes les paires annotées) ===')
print(tab_classiques.to_string(index=False))
print('\n=== Junyi 2015 ===')
print(json.dumps({k: int(v) if isinstance(v, (int, np.integer)) else v for k, v in stats_junyi.items()}, indent=1))
if 'tab_pilote' in globals():
    print('\n=== Pilote LLM (300 paires) ===')
    print(tab_pilote.to_string(index=False))